# 04. 품질 검사 · 문서 선별 · 공통 스냅샷
PR #29의 두 번째 묶음입니다. 가격 검사, 공시 문서 선별, '사업의 내용' 절 탐지, 두 Agent 공통 스냅샷과 저장 차단을 봅니다.

DB·API 키 없이 실행됩니다. 위에서부터 차례로 실행하세요(메뉴 Run → Run All Cells).
예시 자료는 합성·모의 데이터입니다. 실제 기업 분석 결과가 아닙니다.

In [1]:
import json
import sys
import tempfile
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'agent/neon_agents.py').exists()), None)
if ROOT is None:
    raise RuntimeError('압축 전체를 해제한 프로젝트/notebooks에서 실행하세요.')
sys.path.insert(0, str(ROOT))
pd.set_option('display.max_colwidth', 120)
print('프로젝트 위치 확인 완료')

프로젝트 위치 확인 완료


## 1. 가격 검사
- OHLC 논리: 고가 ≥ 시가·종가 ≥ 저가 > 0
- 전일 대비 ±30%(KRX 가격제한폭) 초과: 기업행위로 설명되지 않으면 보류
- 거래일 달력·기업행위 자료는 제공할 때만 대조(없으면 '일부 가능')

In [2]:
from agent.price_checks import check_prices


def rows(closes):
    days = ['2026-10-26', '2026-10-27', '2026-10-28']
    return [{'day': d, 'open': c, 'high': c + 1, 'low': c - 1, 'close': c, 'volume': 100} for d, c in zip(days, closes)]

cases = {
    '정상': (rows([100, 101, 102]), {}),
    'OHLC 오류(고가<저가)': ([{**rows([100, 101, 102])[0], 'high': 90}] + rows([100, 101, 102])[1:], {}),
    '설명 안 되는 +40% 급변': (rows([100, 140, 141]), {}),
    '분할일과 일치하는 -50%': (rows([100, 50, 51]), {'corporate_actions': [
        {'ticker': 'DEMO', 'date': '2026-10-27', 'type': 'split', 'source': '예시 공시'}], 'ticker': 'DEMO'}),
    '달력 대비 누락 거래일': ([r for r in rows([100, 101, 102]) if r['day'] != '2026-10-27'],
                       {'trading_dates': ['2026-10-26', '2026-10-27', '2026-10-28']}),
}
table = []
for name, (data, options) in cases.items():
    for c in check_prices(data, **options):
        table.append({'사례': name, '규칙': c['rule'], '판정': c['status'], '이유': c['reason']})
display(pd.DataFrame(table))

,사례,규칙,판정,이유
0,정상,ohlc_logic,pass,OHLC 논리 위반 없음(전체 필드 있는 행 기준)
1,정상,daily_limit,pass,전일 대비 ±30% 초과 변동 없음
2,정상,corporate_actions,partial,기업행위(분할·병합·권리락 등) 자료 미제공: 대조하지 않음
3,정상,calendar,partial,공식 거래일 달력 미제공: 누락 거래일 판정 안 함
4,OHLC 오류(고가<저가),ohlc_logic,hold,OHLC 논리 위반(고가≥시가·종가≥저가>0)
5,OHLC 오류(고가<저가),daily_limit,pass,전일 대비 ±30% 초과 변동 없음
6,OHLC 오류(고가<저가),corporate_actions,partial,기업행위(분할·병합·권리락 등) 자료 미제공: 대조하지 않음
7,OHLC 오류(고가<저가),calendar,partial,공식 거래일 달력 미제공: 누락 거래일 판정 안 함
8,설명 안 되는 +40% 급변,ohlc_logic,pass,OHLC 논리 위반 없음(전체 필드 있는 행 기준)
9,설명 안 되는 +40% 급변,daily_limit,hold,전일 대비 ±30% 초과 변동: 기업행위 자료로 설명되지 않음


## 2. 공시 문서 선별
제목 규칙: 사업보고서 > 반기보고서 > 분기보고서 > IR, 기업당 최대 3건. 감사보고서·주요사항·뉴스는 제외 사유와 함께 표시합니다.

In [3]:
from agent.document_selection import select_documents

docs = [
    {'doc_id': 1, 'title': '분기보고서 (2026.09)', 'published_at': '2026-11-14', 'source': 'dart'},
    {'doc_id': 2, 'title': '사업보고서 (2025.12)', 'published_at': '2026-03-10', 'source': 'dart'},
    {'doc_id': 3, 'title': '[기재정정]사업보고서 (2025.12)', 'published_at': '2026-04-02', 'source': 'dart'},
    {'doc_id': 4, 'title': '반기보고서 (2026.06)', 'published_at': '2026-08-14', 'source': 'dart'},
    {'doc_id': 5, 'title': '2026 3분기 IR 자료', 'published_at': '2026-10-30', 'source': 'ir'},
    {'doc_id': 6, 'title': '감사보고서 (2025.12)', 'published_at': '2026-03-05', 'source': 'dart'},
    {'doc_id': 7, 'title': '주가 급등 뉴스', 'published_at': '2026-10-01', 'source': 'dart'},
]
selected, skipped = select_documents(docs)
print('선택된 문서')
display(pd.DataFrame(selected)[['doc_id', 'title', 'documentKind', 'priority']])
print('제외된 문서')
display(pd.DataFrame(skipped))

선택된 문서


,doc_id,title,documentKind,priority
0,3,[기재정정]사업보고서 (2025.12),annual,0
1,4,반기보고서 (2026.06),semiannual,1
2,1,분기보고서 (2026.09),quarterly,2


제외된 문서


,doc_id,title,reason
0,2,사업보고서 (2025.12),같은 기간의 최신 정정본으로 대체
1,6,감사보고서 (2025.12),사업 원문 대상 아님(감사·주요사항·첨부 등)
2,7,주가 급등 뉴스,제목 규칙 불일치(사업·반기·분기보고서/IR 아님)
3,5,2026 3분기 IR 자료,우선순위 상위 3건 밖


## 3. '사업의 내용' 절 탐지
정기보고서는 'II. 사업의 내용'(SEC: Item 1. Business) 안의 조각만 근거로 씁니다. 목차는 건너뜁니다.
경쟁 표는 키워드가 맞은 **후보 발췌**이며 경쟁 강도 평가가 아닙니다.

In [4]:
from agent.business_sections import competition_candidates, section_chunks

lines = ['목 차', 'II. 사업의 내용 ........ 5', 'III. 재무에 관한 사항 ........ 30',
         'II. 사업의 내용', '1. 사업의 개요', '당사는 부품을 제조합니다. ' * 30,
         '2. 주요 제품 및 서비스', '주요 고객은 완성품 업체이며 경쟁사와 가격 경쟁을 합니다.',
         '3. 원재료 및 생산설비', '주요 원재료는 외부 공급업체에서 매입합니다.',
         'III. 재무에 관한 사항', '재무 내용(근거에서 제외)']
chunks = [{'doc_id': 1, 'chunk_id': 100 + i, 'chunk_idx': i, 'content': t} for i, t in enumerate(lines)]
found, inside = section_chunks(chunks)
print('탐지 결과:', found['status'], found['standard'], '| 시작:', found['startHeading'], '| 끝:', found['endHeading'])
print('하위 절:', [s['title'] for s in found['subsections']], '| 조각 범위:', found['chunkRange'])
display(pd.DataFrame([{'chunk_idx': c['chunk_idx'], '하위 절': c['section']['subsection'], '내용': c['content'][:60]}
                      for c in inside]))
evidence = [{'evidenceId': f"E{c['chunk_idx']}", 'excerpt': c['content']} for c in inside]
display(pd.DataFrame(competition_candidates(evidence))[['factor', 'assessment', 'evidenceIds']])

탐지 결과: found DART | 시작: II. 사업의 내용 | 끝: III. 재무에 관한 사항
하위 절: ['1. 사업의 개요', '2. 주요 제품 및 서비스', '3. 원재료 및 생산설비'] | 조각 범위: [3, 9]


,chunk_idx,하위 절,내용
0,3,None,II. 사업의 내용
1,4,1. 사업의 개요,1. 사업의 개요
2,5,1. 사업의 개요,당사는 부품을 제조합니다. 당사는 부품을 제조합니다. 당사는 부품을 제조합니다. 당사는 부품을 제조합니다.
3,6,2. 주요 제품 및 서비스,2. 주요 제품 및 서비스
4,7,2. 주요 제품 및 서비스,주요 고객은 완성품 업체이며 경쟁사와 가격 경쟁을 합니다.
5,8,3. 원재료 및 생산설비,3. 원재료 및 생산설비
6,9,3. 원재료 및 생산설비,주요 원재료는 외부 공급업체에서 매입합니다.


,factor,assessment,evidenceIds
0,산업 내 경쟁,"후보 발췌 1건(키워드 일치, 관련성·강도 평가 아님)",[E7]
1,진입 장벽,자료 미확보,[]
2,대체재,자료 미확보,[]
3,고객 교섭력,"후보 발췌 1건(키워드 일치, 관련성·강도 평가 아님)",[E7]
4,공급자 교섭력,"후보 발췌 2건(키워드 일치, 관련성·강도 평가 아님)","[E8, E9]"


## 4. 공통 스냅샷
실제 사용 시에는 `fetch_snapshot()`이 두 Agent 입력을 한 읽기 전용 트랜잭션에서 함께 읽습니다. 여기서는 모의 스냅샷을 씁니다.
같은 스냅샷을 쓰면 두 Agent 결과의 입력 digest가 같아져 서로 비교할 수 있습니다.

In [5]:
from agent.basic_agents import AgentSettings, IndustryBusinessAgent, TechnicalAnalysisAgent, compare_results
from agent.neon_agents import execute_report
from agent.snapshots import load_snapshot, mock_snapshot, reports_from_snapshot, save_snapshot

snapshot = mock_snapshot()
print(snapshot['notice'])
report = reports_from_snapshot(snapshot)[0]
settings = AgentSettings(window=20)
options = {'reference_date': '2026-10-28', 'trading_dates': snapshot['mockTradingDates'], 'corporate_actions': []}
_, industry = execute_report(report, IndustryBusinessAgent(settings=settings), 'industry_business_model', **options)
_, technical = execute_report(report, TechnicalAnalysisAgent(settings=settings), 'technical_analysis', **options)
print('산업 입력 digest:', industry['inputDigest'][:16])
print('기술 입력 digest:', technical['inputDigest'][:16])
print(compare_results(industry, technical)['summary'])
display(pd.DataFrame(report['sectionDetection']))

모의 입력: 실제 회사·시세 자료가 아닙니다. 코드 동작 확인용입니다.
산업 입력 digest: 5b1da621dde2671e
기술 입력 digest: 5b1da621dde2671e
사업 가설과 가격 관찰은 기간·근거가 다릅니다. 투표나 종합점수로 합치지 않습니다.


,doc_id,title,status,standard,heading,endHeading,chunkRange,subsections
0,9001,사업보고서 (2025.12) [모의],found,DART,II. 사업의 내용,III. 재무에 관한 사항,"[6, 12]","[1. 사업의 개요, 2. 주요 제품 및 서비스, 3. 원재료 및 생산설비]"


In [6]:
with tempfile.TemporaryDirectory() as folder:
    path = save_snapshot(snapshot, folder)
    loaded, _ = load_snapshot(path)
    print('저장 후 다시 읽기: digest 일치 =', loaded['digest'] == snapshot['digest'])
    data = json.loads(path.read_text(encoding='utf-8'))
    data['bundles'][0]['prices'][0]['close'] = 1  # 파일을 몰래 수정
    path.write_text(json.dumps(data), encoding='utf-8')
    try:
        load_snapshot(path)
    except ValueError as error:
        print('수정된 파일 읽기 거부:', error)

저장 후 다시 읽기: digest 일치 = True
수정된 파일 읽기 거부: 스냅샷 digest 불일치: 파일이 변경되었거나 손상되었습니다.


## 5. 저장 차단
Verifier 결과가 하나라도 `failed`이면 결과를 저장하지 않습니다.

In [7]:
from agent.result_verifier import verify_result
from agent.snapshots import save_results

good = verify_result(report, technical, settings)
broken = json.loads(json.dumps(technical))
broken['calculations'][0]['result'] = 0
bad = verify_result(report, broken, settings)
print('정상 결과:', good['status'], '/ 변조 결과:', bad['status'])
with tempfile.TemporaryDirectory() as folder:
    print(save_results(folder, 'technical_analysis', {'results': [broken]}, [good, bad])[1])
    print(save_results(folder, 'technical_analysis', {'results': [technical]}, [good])[1])

정상 결과: review_required / 변조 결과: failed
저장 차단: Verifier failed 1건. 오류를 수정하고 처음부터 다시 실행하세요.
저장 완료: results_20261028_061629.json
